# Tickers Closeness Research

This notebook studies how to define closeness between Polymarket markets. The goal is practical: if we later build an encoder that uses context markets `B` for a target market `A`, we need a retrieval rule that can surface the right neighbors.

We compare several closeness signals and evaluate them with weak labels such as `same_family` and `same_domain`.


## Closeness candidates

We will look at a few simple candidate scores:

- **Text cosine**: TF-IDF cosine similarity over question + description + tags.
- **Tag overlap**: Jaccard similarity over tag sets.
- **Resolution-source match**: whether markets point to the same resolution source.
- **Time overlap**: whether markets live during similar periods.
- **Return correlation**: correlation of resampled probability changes over overlapping history.
- **Combined score**: a simple weighted combination of the signals above.

Weak labels:

- **same_family**: a heuristic family id built from domain, tags, and normalized question text.
- **same_domain**: whether the markets share the same coarse domain.

The point is not to claim that these weak labels are perfect. The point is to understand which similarity signals are most useful for context retrieval.


## Environment and Imports

This cell imports the notebook dependencies and the benchmark helpers used to load markets and probability histories.

The notebook is deliberately diagnostic: before designing a learned retriever, we first want to understand which primitive closeness signals are even worth using.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from polymarket_research import PolymarketDataset
from polymarket_research.utils import setup_root

REPO_ROOT = setup_root()

from benchmarks.benchmark_utils import (
    build_repricing_dataset,
    load_snapshot_frame,
    make_feature_matrix,
    make_rolling_splits,
    prepare_resolved_markets,
)
from benchmarks.covariate_utils import load_covariate_config, merge_covariates_asof


## Configuration and Similarity Helpers

This cell defines the working domains and helper functions for weak labels and raw similarity signals.

The weak family construction is heuristic on purpose. We are not claiming perfect ground truth for neighbor relations; we only need a practical proxy to compare closeness definitions.


In [ ]:
DOMAINS = (
    # 'crypto',
     'politics', 'geopolitics', 'technology', 'finance_economy')
MAX_MARKETS_PER_DOMAIN = 100
MIN_PROBABILITY_ROWS = 288
RESAMPLE_FREQ = '1h'
TOP_K = 5


def parse_listish(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except Exception:
        pass
    if '|' in text:
        parts = text.split('|')
    elif ',' in text:
        parts = text.split(',')
    else:
        parts = [text]
    return [part.strip() for part in parts if part.strip()]


def normalize_text(value: str) -> str:
    text = str(value or '').lower()
    text = re.sub(r'[^a-z0-9\s]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def build_family_id(question: str, domain: str, tags) -> str:
    norm_q = normalize_text(question)
    norm_tags = [normalize_text(tag) for tag in parse_listish(tags)]
    tokens = [tok for tok in norm_q.split() if tok not in {'will', 'the', 'a', 'an', 'be', 'is', 'are', 'to', 'of', 'by', 'in'}]
    key = ' '.join(tokens[:6]) if tokens else norm_q[:48]
    tag_key = '|'.join(sorted(norm_tags[:3]))
    return f"{domain}::{tag_key}::{key}".strip(':')


def tag_jaccard(tags_a, tags_b):
    a = set(parse_listish(tags_a))
    b = set(parse_listish(tags_b))
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)


def safe_auc(y_true, score):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, score)


def safe_ap(y_true, score):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return average_precision_score(y_true, score)


def safe_corr(series_a: pd.Series, series_b: pd.Series, min_points: int = 12) -> float:
    pair = pd.concat([series_a, series_b], axis=1).dropna()
    if len(pair) < min_points:
        return np.nan
    a = pair.iloc[:, 0].to_numpy(dtype=float)
    b = pair.iloc[:, 1].to_numpy(dtype=float)
    if np.nanstd(a) <= 1e-12 or np.nanstd(b) <= 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def recall_at_k(pair_df: pd.DataFrame, score_col: str, label_col: str, k: int = 5) -> float:
    recalls = []
    for market_id, group in pair_df.groupby('market_id_a', sort=False):
        positives = int(group[label_col].sum())
        if positives == 0:
            continue
        top = group.sort_values(score_col, ascending=False).head(k)
        recalls.append(float(top[label_col].sum() > 0))
    return float(np.mean(recalls)) if recalls else np.nan


## Load Markets and Build Weak Labels

This cell loads markets across domains, fetches their histories, and constructs heuristic metadata such as `family_id` and a text blob.

This gives us the objects whose closeness we want to study. The output of the notebook is not a final graph; it is evidence about which retrieval signals are worth keeping.


In [ ]:
DATASET_ARTEFACT_DIR = REPO_ROOT / 'research_notebooks' / 'running_artefacts'

dataset = PolymarketDataset.from_parquet(DATASET_ARTEFACT_DIR)
markets = dataset.markets.copy()
probabilities = dataset.probabilities.copy()


## Construct Pairwise Similarity Signals

Here we build the pairwise market table and compute the candidate closeness signals:

- text cosine
- tag overlap
- time overlap
- resolution-source match
- return correlation
- a simple combined score

This is the core research object of the notebook. If a signal cannot recover plausible neighbors even under weak labels, it is unlikely to be useful later inside a context encoder.


In [ ]:
vectorizer = TfidfVectorizer(min_df=2, max_features=5000, ngram_range=(1, 2), stop_words='english')
tfidf = vectorizer.fit_transform(markets['text_blob'].fillna(''))
text_cos = cosine_similarity(tfidf)

meta = markets.set_index('market_id')
prob_hourly = (
    probabilities[['market_id', 'timestamp_utc', 'yes_probability']]
    .dropna()
    .assign(timestamp_utc=lambda x: pd.to_datetime(x['timestamp_utc'], utc=True, errors='coerce'))
    .set_index('timestamp_utc')
    .groupby('market_id')['yes_probability']
    .resample(RESAMPLE_FREQ)
    .last()
    .reset_index()
)
wide_prob = prob_hourly.pivot(index='timestamp_utc', columns='market_id', values='yes_probability').sort_index()
wide_ret = wide_prob.diff()

pair_rows = []
market_ids = markets['market_id'].tolist()
for i, j in itertools.combinations(range(len(market_ids)), 2):
    mid_a = market_ids[i]
    mid_b = market_ids[j]
    row_a = markets.iloc[i]
    row_b = markets.iloc[j]
    shared = wide_ret[[mid_a, mid_b]]
    corr = safe_corr(shared[mid_a], shared[mid_b], min_points=12)
    overlap_start = max(row_a['created_at'], row_b['created_at'])
    overlap_end = min(row_a['end_date'], row_b['end_date'])
    overlap_hours = max(0.0, (overlap_end - overlap_start).total_seconds() / 3600.0)
    union_hours = max((max(row_a['end_date'], row_b['end_date']) - min(row_a['created_at'], row_b['created_at'])).total_seconds() / 3600.0, 1.0)
    time_overlap = overlap_hours / union_hours
    source_match = float(str(row_a['resolution_source']) == str(row_b['resolution_source']) and pd.notna(row_a['resolution_source']))
    tag_score = tag_jaccard(row_a['tag_labels'], row_b['tag_labels'])
    same_family = float(row_a['family_id'] == row_b['family_id'])
    same_domain = float(row_a['primary_domain'] == row_b['primary_domain'])
    pair_rows.append({
        'market_id_a': mid_a,
        'market_id_b': mid_b,
        'question_a': row_a['question'],
        'question_b': row_b['question'],
        'domain_a': row_a['primary_domain'],
        'domain_b': row_b['primary_domain'],
        'same_family': same_family,
        'same_domain': same_domain,
        'text_cosine': float(text_cos[i, j]),
        'tag_jaccard': float(tag_score),
        'time_overlap': float(time_overlap),
        'source_match': source_match,
        'return_corr': corr,
    })

pairs = pd.DataFrame(pair_rows)
pairs['return_corr_abs'] = pairs['return_corr'].abs()
pairs['combined_score'] = (
    0.45 * pairs['text_cosine'].fillna(0.0) +
    0.20 * pairs['tag_jaccard'].fillna(0.0) +
    0.15 * pairs['time_overlap'].fillna(0.0) +
    0.10 * pairs['source_match'].fillna(0.0) +
    0.10 * pairs['return_corr_abs'].fillna(0.0)
)
pairs.head(3)


## Retrieval Metrics Under Weak Labels

This cell turns pairwise closeness scores into evaluation metrics such as ROC-AUC, average precision, and recall@k.

The logic is that a useful neighbor score should not only separate positive pairs globally, but also surface at least one good neighbor near the top of the retrieval list for a given market.


In [ ]:
score_cols = ['text_cosine', 'tag_jaccard', 'time_overlap', 'source_match', 'return_corr_abs', 'combined_score']
metric_rows = []
for label_col in ['same_family', 'same_domain']:
    for score_col in score_cols:
        metric_rows.append({
            'label': label_col,
            'score': score_col,
            'roc_auc': safe_auc(pairs[label_col], pairs[score_col].fillna(0.0)),
            'avg_precision': safe_ap(pairs[label_col], pairs[score_col].fillna(0.0)),
            f'recall_at_{TOP_K}': recall_at_k(pairs, score_col, label_col, k=TOP_K),
        })

metrics = pd.DataFrame(metric_rows)
display(metrics.sort_values(['label', 'roc_auc'], ascending=[True, False]))


## Metric Summary Plot

This plot compares the candidate closeness scores as retrieval mechanisms.

At this stage we care less about one decimal point and more about ranking the raw channels: which signals are consistently useful, which are weak alone, and which might belong in a combined retriever.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.barplot(data=metrics, x='roc_auc', y='score', hue='label', ax=axes[0], palette='crest')
axes[0].set_title('Pairwise ROC-AUC for Weak Retrieval Labels')
axes[0].set_xlabel('Higher is better')
axes[0].set_ylabel('')

sns.barplot(data=metrics, x=f'recall_at_{TOP_K}', y='score', hue='label', ax=axes[1], palette='flare')
axes[1].set_title(f'Neighbor Recall@{TOP_K} for Weak Retrieval Labels')
axes[1].set_xlabel('Higher is better')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()


## Distribution and Interaction Diagnostics

These plots show how a single closeness score behaves by weak label and how two different channels relate to each other.

This is useful for deciding whether two signals are redundant or complementary. For example, strong text similarity and strong return correlation together suggest a more robust notion of relatedness than either signal alone.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.histplot(data=pairs, x='text_cosine', hue='same_family', stat='density', common_norm=False, bins=40, ax=axes[0])
axes[0].set_title('Text cosine by same-family label')

sns.scatterplot(
    data=pairs.sample(min(len(pairs), 5000), random_state=0),
    x='text_cosine',
    y='return_corr_abs',
    hue='same_family',
    alpha=0.6,
    ax=axes[1],
)
axes[1].set_title('Text similarity vs return correlation')
axes[1].set_xlabel('Text cosine')
axes[1].set_ylabel('|Return correlation|')
plt.tight_layout()
plt.show()


## Qualitative Neighbor Inspection

Metrics alone are not enough. This cell shows nearest neighbors for a few sample markets under different scores.

That qualitative check matters because a score can look numerically decent while still retrieving semantically odd or practically useless neighbors.


In [ ]:
def nearest_neighbors(pair_df: pd.DataFrame, market_id: str, score_col: str, top_k: int = 5) -> pd.DataFrame:
    left = pair_df.loc[pair_df['market_id_a'] == market_id, ['market_id_b', 'question_b', score_col, 'same_family', 'same_domain']].copy()
    left = left.rename(columns={'market_id_b': 'neighbor_id', 'question_b': 'neighbor_question'})
    right = pair_df.loc[pair_df['market_id_b'] == market_id, ['market_id_a', 'question_a', score_col, 'same_family', 'same_domain']].copy()
    right = right.rename(columns={'market_id_a': 'neighbor_id', 'question_a': 'neighbor_question'})
    out = pd.concat([left, right], ignore_index=True)
    return out.sort_values(score_col, ascending=False).head(top_k)


example_ids = markets.groupby('primary_domain').head(2)['market_id'].tolist()
for market_id in example_ids:
    question = meta.loc[market_id, 'question']
    print('\n' + '=' * 100)
    print('QUERY:', question)
    print('- top neighbors by text_cosine')
    display(nearest_neighbors(pairs, market_id, 'text_cosine', top_k=5))
    print('- top neighbors by combined_score')
    display(nearest_neighbors(pairs, market_id, 'combined_score', top_k=5))


## Closeness Heatmap

The heatmap gives a compact visual view of the local market graph induced by the combined score.

This is helpful for spotting whether the score creates coherent clusters or whether it mostly behaves like noise.


In [ ]:
heatmap_ids = markets.sort_values(['primary_domain', 'family_id', 'market_id']).head(20)['market_id'].tolist()
heat_pairs = pairs.loc[pairs['market_id_a'].isin(heatmap_ids) & pairs['market_id_b'].isin(heatmap_ids), ['market_id_a', 'market_id_b', 'combined_score']].copy()
heat = pd.DataFrame(np.eye(len(heatmap_ids)), index=heatmap_ids, columns=heatmap_ids)
for row in heat_pairs.itertuples(index=False):
    heat.loc[row.market_id_a, row.market_id_b] = row.combined_score
    heat.loc[row.market_id_b, row.market_id_a] = row.combined_score

plt.figure(figsize=(10, 8))
sns.heatmap(heat, cmap='mako', square=True)
plt.title('Combined closeness heatmap for a sample of markets')
plt.xlabel('Market id')
plt.ylabel('Market id')
plt.tight_layout()
plt.show()


## How to use this notebook

A good closeness score should do at least one of these well:

- recover weakly related markets such as `same_family`
- retrieve semantically and temporally plausible neighbors
- provide a useful candidate set for later `B` retrieval in the encoder notebook

A reasonable next step is to replace the hand-crafted combined score with a learned retrieval module, but only after this notebook tells us which raw similarity channels are actually useful.
